# 🥇 Gold — tables for the dashboard

Two business tables from `my.silver.sales`:

- **`my.gold.sales_daily`** — orders, units and revenue per day, country, category and channel.
  Every column adds up, so one table serves all the dashboard's charts and filters.
- **`my.gold.top_products`** — revenue and units per product, ranked.

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS my.gold")

spark.sql("""
    CREATE OR REPLACE TABLE my.gold.sales_daily AS
    SELECT order_date, country, category, channel,
           count(*)      AS orders,
           sum(quantity) AS units,
           sum(revenue)  AS revenue
    FROM my.silver.sales
    GROUP BY order_date, country, category, channel""")

spark.sql("""
    CREATE OR REPLACE TABLE my.gold.top_products AS
    SELECT product, category,
           count(*)      AS orders,
           sum(quantity) AS units,
           sum(revenue)  AS revenue,
           rank() OVER (ORDER BY sum(revenue) DESC) AS revenue_rank
    FROM my.silver.sales
    GROUP BY product, category""")

print("my.gold.sales_daily:", spark.table("my.gold.sales_daily").count(), "rows")
print("my.gold.top_products:", spark.table("my.gold.top_products").count(), "rows")

## Revenue per month

In [ ]:
%%sql
SELECT CAST(date_trunc('month', order_date) AS date) AS month,
       sum(orders) AS orders, sum(revenue) AS revenue
FROM my.gold.sales_daily
GROUP BY 1
ORDER BY 1

## Top 5 products

In [ ]:
%%sql
SELECT revenue_rank, product, category, units, revenue
FROM my.gold.top_products
ORDER BY revenue_rank
LIMIT 5